In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/fatihitsbatulhaq/deep-armocromia/Deep Armocromia/release/RGB.zip
/kaggle/input/datasets/fatihitsbatulhaq/deep-armocromia/Deep Armocromia/release/BM.zip
/kaggle/input/datasets/fatihitsbatulhaq/deep-armocromia/Deep Armocromia/release/RGB-M.zip
/kaggle/input/datasets/fatihitsbatulhaq/deep-armocromia/Deep Armocromia/release/annotations.csv
/kaggle/input/datasets/fatihitsbatulhaq/deep-armocromia/Deep Armocromia/release/annotations.xlsx
/kaggle/input/datasets/fatihitsbatulhaq/deep-armocromia/Deep Armocromia/original_imager_facer_mask/ORIGINAL_RGB_NOT_PROCESSED.zip
/kaggle/input/datasets/fatihitsbatulhaq/deep-armocromia/Deep Armocromia/original_imager_facer_mask/facer_masks.zip


In [2]:
print("Stage 3 Kaggle → GitHub test")

Stage 3 Kaggle → GitHub test


In [3]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")

print("GitHub token loaded:", bool(GITHUB_TOKEN))

GitHub token loaded: True


In [4]:
import requests
import importlib
from kaggle_secrets import UserSecretsClient

# 1. Ambil token
user_secrets = UserSecretsClient()
GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")

REPO_OWNER = "fatih-collab"
REPO_NAME = "model-lab"
FILE_PATH = "scripts/register_experiment.py"

# 2. Unduh raw file dari GitHub menggunakan API
url = f"https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}/contents/{FILE_PATH}"
headers = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3.raw"
}

response = requests.get(url, headers=headers)
if response.status_code == 200:
    with open("register_experiment.py", "w") as f:
        f.write(response.text)
    print("PASS: register_experiment.py berhasil diunduh ke environment Kaggle.")
else:
    print(f"FAIL: {response.status_code} - {response.text}")

# 3. Import modul dan test fungsi generate_experiment_id
import register_experiment

# Reload modul (berguna jika Anda menjalankan sel ini berkali-kali)
importlib.reload(register_experiment)

test_id = register_experiment.generate_experiment_id()
print(f"PASS: Modul ter-load. Test ID: {test_id}")

PASS: register_experiment.py berhasil diunduh ke environment Kaggle.
PASS: Modul ter-load. Test ID: EXP-20261005-014055-015c


In [5]:
import json
import base64
from datetime import datetime

def upload_registry_to_github(experiment_data: dict, token: str, repo_owner: str, repo_name: str) -> bool:
    """
    Mengunggah registry experiment ke GitHub tanpa mengeksekusi trigger secara langsung.
    """
    # Validasi menggunakan source of truth yang baru saja diunduh
    try:
        register_experiment.validate_experiment(experiment_data)
        print("PASS: Validasi lokal berhasil.")
    except Exception as e:
        print(f"FAIL: Validasi gagal - {e}")
        return False

    topic = experiment_data["topic"]
    exp_id = experiment_data["experiment_id"]
    file_path = f"registry/{topic}/{exp_id}.json"
    
    url = f"https://api.github.com/repos/{repo_owner}/{repo_name}/contents/{file_path}"
    
    # Serialize data ke JSON string sesuai format register_experiment.py
    json_content = json.dumps(experiment_data, indent=2, ensure_ascii=False) + "\n"
    encoded_content = base64.b64encode(json_content.encode("utf-8")).decode("utf-8")
    
    headers = {
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github.v3+json"
    }
    
    payload = {
        "message": f"Kaggle: Register experiment {exp_id}",
        "content": encoded_content,
        "branch": "main"  # Pastikan branch ini sesuai dengan repo Anda
    }
    
    print(f"Mengunggah {file_path} ke GitHub...")
    response = requests.put(url, headers=headers, json=payload)
    
    if response.status_code in [200, 201]:
        print(f"PASS: File {file_path} berhasil diunggah!")
        return True
    else:
        print(f"FAIL: Upload gagal [{response.status_code}] - {response.json().get('message')}")
        return False

In [6]:
import datetime

# 1. Buat ID baru
exp_id = register_experiment.generate_experiment_id()
now_iso = datetime.datetime.now(datetime.timezone.utc).astimezone().isoformat()

# 2. Susun payload sesuai skema validate_experiment
dummy_experiment = {
    "experiment_id": exp_id,
    "topic": "topic-a",
    "model": "XGBoost-Test-E2E",
    "objective": "Verifikasi pipeline Kaggle ke GitHub Actions",
    "dataset": "dataset-v1",
    "created_at": now_iso,
    "metrics": {
        "accuracy": 0.99,
        "f1": 0.98
    },
    "status": "SUCCESS",
    "owner": "Fatih Itsbatul Haq",
    "kaggle": {
        "version": "1",
        "url": "https://kaggle.com/fatih/model-lab-test"
    },
    "notebook": "EXP-test-notebook.ipynb",
    "previous_experiment": None
}

# 3. Eksekusi fungsi helper
upload_registry_to_github(
    experiment_data=dummy_experiment,
    token=GITHUB_TOKEN,
    repo_owner=REPO_OWNER,
    repo_name=REPO_NAME
)

PASS: Validasi lokal berhasil.
Mengunggah registry/topic-a/EXP-20261005-014055-afc1.json ke GitHub...
PASS: File registry/topic-a/EXP-20261005-014055-afc1.json berhasil diunggah!


True

In [7]:
import json
from sklearn.metrics import accuracy_score, f1_score

# DATA DUMMY: Ditambahkan agar tidak terjadi NameError
y_test = [0, 1, 1, 0, 1, 1, 0, 1]
y_pred = [0, 1, 0, 0, 1, 1, 0, 1]

experiment_metadata = {
    "topic": "topic-a",
    "model": "XGBoost-Tuning",
    "objective": "Verifikasi Pipeline Payload",
    "dataset": "dataset-v2",
    "owner": "Fatih Itsbatul Haq",
    "metrics": {
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "f1": float(f1_score(y_test, y_pred, average='macro'))
    }
}

print("===REGISTRY_PAYLOAD_START===")
print(json.dumps(experiment_metadata))
print("===REGISTRY_PAYLOAD_END===")

===REGISTRY_PAYLOAD_START===
{"topic": "topic-a", "model": "XGBoost-Tuning", "objective": "Verifikasi Pipeline Payload", "dataset": "dataset-v2", "owner": "Fatih Itsbatul Haq", "metrics": {"accuracy": 0.875, "f1": 0.873015873015873}}
===REGISTRY_PAYLOAD_END===
